# Lab4-Assignment about Named Entity Recognition and Classification

This notebook describes the assignment of Lab 4 of the text mining course. We assume you have succesfully completed Lab1, Lab2 and Lab3 as welll. Especially Lab2 is important for completing this assignment.

**Learning goals**
* going from linguistic input format to representing it in a feature space
* working with pretrained word embeddings
* train a supervised classifier (SVM)
* evaluate a supervised classifier (SVM)
* learn how to interpret the system output and the evaluation results
* be able to propose future improvements based on the observed results


## Credits
This notebook was originally created by Marten Postma and Filip Ilievski and adapted by Piek vossen

## [Points: 18] Exercise 1 (NERC): Training and evaluating an SVM using CoNLL-2003

**[4 point] a) Load the CoNLL-2003 training data using the *ConllCorpusReader* and create for both *train.txt* and *test.txt*:**

    [2 points]  -a list of dictionaries representing the features for each training instances, e..g,
    ```
    [
    {'words': 'EU', 'pos': 'NNP'}, 
    {'words': 'rejects', 'pos': 'VBZ'},
    ...
    ]
    ```

    [2 points] -the NERC labels associated with each training instance, e.g.,
    dictionaries, e.g.,
    ```
    [
    'B-ORG', 
    'O',
    ....
    ]
    ```

In [2]:
import os
from nltk.corpus.reader import ConllCorpusReader

# 注意:解压后多了一层 CONLL2003 文件夹,所以要写两次
DATA_DIR = os.path.join('CONLL2003', 'CONLL2003')

# ---- Training data ----
train = ConllCorpusReader(DATA_DIR, 'train.txt', ['words', 'pos', 'ignore', 'chunk'])

training_features = []
training_gold_labels = []

for token, pos, ne_label in train.iob_words():
    a_dict = {
        'words': token,
        'pos': pos
    }
    training_features.append(a_dict)
    training_gold_labels.append(ne_label)

print("Training instances:", len(training_features))
print("First 3 features:", training_features[:3])
print("First 3 labels:", training_gold_labels[:3])

Training instances: 203621
First 3 features: [{'words': 'EU', 'pos': 'NNP'}, {'words': 'rejects', 'pos': 'VBZ'}, {'words': 'German', 'pos': 'JJ'}]
First 3 labels: ['B-ORG', 'O', 'B-MISC']


In [3]:
# ---- Test data ----
test = ConllCorpusReader(DATA_DIR, 'test.txt', ['words', 'pos', 'ignore', 'chunk'])

test_features = []
test_gold_labels = []

for token, pos, ne_label in test.iob_words():
    a_dict = {
        'words': token,
        'pos': pos
    }
    test_features.append(a_dict)
    test_gold_labels.append(ne_label)

print("Test instances:", len(test_features))
print("First 3 features:", test_features[:3])
print("First 3 labels:", test_gold_labels[:3])

Test instances: 46435
First 3 features: [{'words': 'SOCCER', 'pos': 'NN'}, {'words': '-', 'pos': ':'}, {'words': 'JAPAN', 'pos': 'NNP'}]
First 3 labels: ['O', 'O', 'B-LOC']


**[2 points] b) provide descriptive statistics about the training and test data:**
* How many instances are in train and test?
* Provide a frequency distribution of the NERC labels, i.e., how many times does each NERC label occur?
* Discuss to what extent the training and test data is balanced (equal amount of instances for each NERC label) and to what extent the training and test data differ?

Tip: you can use the following `Counter` functionality to generate frequency list of a list:

In [4]:
from collections import Counter 

my_list=[1,2,1,3,2,5]
Counter(my_list)

print("Number of training instances:", len(training_features))
print("Number of test instances:    ", len(test_features))
print()

# 2. NERC 标签的频率分布
train_label_counts = Counter(training_gold_labels)
test_label_counts  = Counter(test_gold_labels)

print("Training label distribution:")
for label, count in sorted(train_label_counts.items(), key=lambda x: -x[1]):
    print(f"  {label:10s} {count:>7d}  ({count/len(training_gold_labels)*100:5.2f}%)")

print("\nTest label distribution:")
for label, count in sorted(test_label_counts.items(), key=lambda x: -x[1]):
    print(f"  {label:10s} {count:>7d}  ({count/len(test_gold_labels)*100:5.2f}%)")

Number of training instances: 203621
Number of test instances:     46435

Training label distribution:
  O           169578  (83.28%)
  B-LOC         7140  ( 3.51%)
  B-PER         6600  ( 3.24%)
  B-ORG         6321  ( 3.10%)
  I-PER         4528  ( 2.22%)
  I-ORG         3704  ( 1.82%)
  B-MISC        3438  ( 1.69%)
  I-LOC         1157  ( 0.57%)
  I-MISC        1155  ( 0.57%)

Test label distribution:
  O            38323  (82.53%)
  B-LOC         1668  ( 3.59%)
  B-ORG         1661  ( 3.58%)
  B-PER         1617  ( 3.48%)
  I-PER         1156  ( 2.49%)
  I-ORG          835  ( 1.80%)
  B-MISC         702  ( 1.51%)
  I-LOC          257  ( 0.55%)
  I-MISC         216  ( 0.47%)


### Discussion: Descriptive statistics of the CoNLL-2003 train and test data

#### 1. Number of instances
- **Training set**: 203,621 token-level instances
- **Test set**: 46,435 token-level instances
- The test set is roughly **23%** the size of the training set, which is a typical train/test ratio.

#### 2. Frequency distribution of NERC labels

| Label   | Train count | Train %  | Test count | Test %   |
|---------|------------:|---------:|-----------:|---------:|
| O       |     169,578 | 83.28 %  |     38,323 | 82.53 %  |
| B-LOC   |       7,140 |  3.51 %  |      1,668 |  3.59 %  |
| B-PER   |       6,600 |  3.24 %  |      1,617 |  3.48 %  |
| B-ORG   |       6,321 |  3.10 %  |      1,661 |  3.58 %  |
| I-PER   |       4,528 |  2.22 %  |      1,156 |  2.49 %  |
| I-ORG   |       3,704 |  1.82 %  |        835 |  1.80 %  |
| B-MISC  |       3,438 |  1.69 %  |        702 |  1.51 %  |
| I-LOC   |       1,157 |  0.57 %  |        257 |  0.55 %  |
| I-MISC  |       1,155 |  0.57 %  |        216 |  0.47 %  |

#### 3. Are the data balanced? Do train and test differ?

**The data is highly imbalanced.** In both the training and test sets, the `O` label (non-entity tokens) dominates, accounting for **~83%** of all tokens. This is expected for token-level NERC, since most words in natural text are not part of a named entity. The remaining ~17% is split across 8 entity sub-labels (B-/I- × LOC/PER/ORG/MISC), each of which individually accounts for less than 4% of the data. The rarest labels — `I-LOC` and `I-MISC` — make up only around 0.5% each.

**Training and test distributions are very similar.** Comparing the two columns side by side, the percentage of each label is almost identical (e.g., `O` is 83.28% in train vs. 82.53% in test; `B-LOC` is 3.51% vs. 3.59%). This indicates that the train/test split is well stratified and drawn from the same underlying distribution, which is desirable for a fair evaluation. The only minor differences are that **`B-ORG` is slightly more frequent in test** (3.58% vs. 3.10%) and **`I-MISC` is slightly less frequent in test** (0.47% vs. 0.57%), but these differences are small and unlikely to have a major impact on evaluation.


**[2 points] c) Concatenate the train and test features (the list of dictionaries) into one list. Load it using the *DictVectorizer*. Afterwards, split it back to training and test.**

Tip: You’ve concatenated train and test into one list and then you’ve applied the DictVectorizer.
The order of the rows is maintained. You can hence use an index (number of training instances) to split the_array back into train and test. Do NOT use: `
from sklearn.model_selection import train_test_split` here.


In [5]:
from sklearn.feature_extraction import DictVectorizer

In [6]:
vec = DictVectorizer()
combined_features = training_features + test_features
the_array = vec.fit_transform(combined_features)

print("Combined sparse matrix shape:", the_array.shape)

n_train = len(training_features)
train_array = the_array[:n_train]
test_array  = the_array[n_train:]

print("Train array shape:", train_array.shape)
print("Test  array shape:", test_array.shape)

#sanity check
assert train_array.shape[0] == len(training_features)
assert test_array.shape[0]  == len(test_features)
print("size matches.")

Combined sparse matrix shape: (250056, 27361)
Train array shape: (203621, 27361)
Test  array shape: (46435, 27361)
size matches.


**[4 points] d) Train the SVM using the train features and labels and evaluate on the test data. Provide a classification report (sklearn.metrics.classification_report).**
The train (*lin_clf.fit*) might take a while. On my computer, it took 1min 53s, which is acceptable. Training models normally takes much longer. If it takes more than 5 minutes, you can use a subset for training. Describe the results:
* Which NERC labels does the classifier perform well on? Why do you think this is the case?
* Which NERC labels does the classifier perform poorly on? Why do you think this is the case?

In [7]:
from sklearn import svm

In [8]:
lin_clf = svm.LinearSVC()

In [ ]:
from sklearn.metrics import classification_report

# Train
lin_clf.fit(train_array, training_gold_labels)

# Predict
predicted_labels = lin_clf.predict(test_array)

# Evaluate
print(classification_report(test_gold_labels, predicted_labels))

              precision    recall  f1-score   support

       B-LOC       0.81      0.77      0.79      1668
      B-MISC       0.78      0.66      0.71       702
       B-ORG       0.79      0.52      0.63      1661
       B-PER       0.86      0.44      0.58      1617
       I-LOC       0.62      0.53      0.57       257
      I-MISC       0.59      0.59      0.59       216
       I-ORG       0.66      0.48      0.55       835
       I-PER       0.33      0.87      0.48      1156
           O       0.99      0.98      0.98     38323

    accuracy                           0.92     46435
   macro avg       0.71      0.65      0.65     46435
weighted avg       0.94      0.92      0.92     46435



/opt/anaconda3/lib/python3.12/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


#### Which NERC labels does the classifier perform well on?

The classifier performs best on the `O` label, with a precision of 0.99, recall of 0.98, and f1-score of 0.98. This is the strongest result in the report, but it is also partly expected because `O` is by far the most common label. There are 38,323 `O` tokens out of 46,435 total tokens, meaning most words in the dataset are not named entities. Because the model sees so many examples of `O`, it learns this category very well. However, this also makes the overall accuracy of 0.92 somewhat misleading, because the model can get a high score by doing very well on the dominant non-entity class.

Among the actual named entity labels, the classifier performs best on `B-LOC`, with an f1-score of 0.79. It also performs relatively well on `B-MISC`, with an f1-score of 0.71. This may be because beginning labels are easier to identify than inside labels. For example, location names often appear as proper nouns and may have repeated recognizable patterns in the training data. A word like a country, city, or region can often be identified from the word itself and its POS tag. `B-MISC` also does fairly well, although its recall is lower than its precision, meaning the model is more careful when predicting `B-MISC` but misses some true examples.

The classifier also has fairly high precision for `B-PER` and `B-ORG`, with precision scores of 0.86 and 0.79. This means that when the model predicts the beginning of a person or organization, it is often correct. However, their recall scores are much lower, so the model is not finding all true person and organization entities. This suggests that the model is conservative with these labels: it avoids making too many false positive predictions, but as a result, it misses many real entities.

#### Which NERC labels does the classifier perform poorly on? 

The classifier performs poorly on several inside labels, especially `I-PER`, `I-ORG`, and `I-LOC`. `I-PER` has an f1-score of only 0.48, even though its recall is very high at 0.87. The main problem is its precision, which is only 0.33. This means the classifier predicts `I-PER` too often. It successfully catches many true inside-person tokens, but it also incorrectly labels many non-`I-PER` tokens as `I-PER`. This is a sign that the model may be confusing proper nouns or name-like words with the inside of a person name.

`I-ORG` also performs poorly, with an f1-score of 0.55, and `I-LOC` has an f1-score of 0.57. These labels are difficult because inside labels depend heavily on the surrounding sequence. For example, an `I-ORG` label usually only makes sense if the previous token was `B-ORG` or `I-ORG`. But the SVM model classifies each token separately, so it does not fully understand the BIO structure. It may predict an inside label without correctly tracking whether the entity has already started.

The classifier also struggles with `B-PER`, which has an f1-score of 0.58, and `B-ORG`, which has an f1-score of 0.63. Their precision is much higher than their recall, which means the classifier is not randomly guessing these labels, but it misses many actual examples. This may happen because person names and organization names are very diverse. A person name, organization name, or location can all look similar at the token level, especially if they are proper nouns. Without richer context, the model has difficulty deciding which type of entity a word belongs to.

Another reason for weaker performance is class imbalance. Labels like `I-LOC` and `I-MISC` have much smaller support, with only 257 and 216 examples in the test set. Since the model has fewer examples of these labels, it has fewer chances to learn stable patterns. This explains why the macro average f1-score is only 0.65 even though the weighted average f1-score is 0.92. The weighted score is dominated by the very common `O` label, while the macro score shows that the model is much less reliable across the smaller entity classes.

**[6 points] e) Train a model that uses the embeddings of these words as inputs. Test again on the same data as in 2d. Generate a classification report and compare the results with the classifier you built in 2d.**

In [15]:
# your code here
import gensim
import numpy as np

word_embedding_model = gensim.models.KeyedVectors.load_word2vec_format(r'GoogleNews-vectors-negative300.bin', binary=True)

In [16]:
# Create embedding vectors for the training data
train_input_vectors = []
train_embedding_labels = []

for feature_dict, ne_label in zip(training_features, training_gold_labels):
    token = feature_dict['words']
    
    if token in word_embedding_model:
        vector = word_embedding_model[token]
    else:
        vector = [0] * 300
        
    train_input_vectors.append(vector)
    train_embedding_labels.append(ne_label)

# Create embedding vectors for the test data
test_input_vectors = []
test_embedding_labels = []

for feature_dict, ne_label in zip(test_features, test_gold_labels):
    token = feature_dict['words']
    
    if token in word_embedding_model:
        vector = word_embedding_model[token]
    else:
        vector = [0] * 300
        
    test_input_vectors.append(vector)
    test_embedding_labels.append(ne_label)

In [ ]:
# Train 
embedding_clf = svm.LinearSVC()
embedding_clf.fit(train_input_vectors, train_embedding_labels)

# Predict 
embedding_predicted_labels = embedding_clf.predict(test_input_vectors)

# Evaluate
print(classification_report(test_embedding_labels, embedding_predicted_labels))

              precision    recall  f1-score   support

       B-LOC       0.76      0.80      0.78      1668
      B-MISC       0.72      0.70      0.71       702
       B-ORG       0.69      0.64      0.66      1661
       B-PER       0.75      0.67      0.71      1617
       I-LOC       0.51      0.42      0.46       257
      I-MISC       0.60      0.54      0.57       216
       I-ORG       0.48      0.33      0.39       835
       I-PER       0.59      0.50      0.54      1156
           O       0.97      0.99      0.98     38323

    accuracy                           0.93     46435
   macro avg       0.68      0.62      0.64     46435
weighted avg       0.92      0.93      0.92     46435



### Comparison

The embedding-based classifier in part e has a slightly higher overall accuracy than the part d classifier, 0.93 compared to 0.92. However, this does not necessarily mean the embedding model is clearly better. The `O` label dominates the dataset, with 38,323 examples out of 46,435 total test tokens, so small improvements on `O` can increase the overall accuracy even if some named entity labels get worse. Both models have the same weighted average f1-score of 0.92, while the macro average f1-score actually drops slightly from 0.65 in part d to 0.64 in part e. This suggests that the embedding model does not improve performance evenly across all labels.

The biggest improvement from using embeddings is on beginning entity labels, especially `B-PER` and `B-ORG`. For `B-PER`, the f1-score increases from 0.58 to 0.71, and recall increases from 0.44 to 0.67. For `B-ORG`, the f1-score increases from 0.63 to 0.66, and recall increases from 0.52 to 0.64. This suggests that embeddings help the model identify more true person and organization beginnings. A likely reason is that word embeddings represent words based on semantic and contextual similarity, so the model is not only relying on exact word/POS patterns. If a name or organization was not strongly represented in the training data, embeddings may still place it near similar words, allowing the classifier to generalize better.

However, this improvement comes with a tradeoff. For most beginning labels, precision decreases in the embedding model. For example, `B-PER` precision drops from 0.86 to 0.75, and `B-ORG` precision drops from 0.79 to 0.69. This means the embedding classifier finds more true entities, but it also makes more false positive predictions. In other words, the model becomes less conservative. The part d classifier was more careful: when it predicted `B-PER` or `B-ORG`, it was often right, but it missed many real examples. The embedding model predicts these labels more often, which improves recall but lowers precision.

The embedding model performs similarly on `B-LOC` and `B-MISC`. `B-LOC` changes only slightly, from an f1-score of 0.79 to 0.78, and `B-MISC` stays at 0.71. This suggests that embeddings do not add much improvement for these labels. For locations, the part d classifier may already perform well because locations often have recognizable word-level patterns, such as country or city names, and many may appear in the training data. Since the DictVectorizer model can use exact word identity, it may already capture many location names effectively.

The embedding model performs worse on several inside labels, especially `I-LOC` and `I-ORG`. `I-LOC` drops from an f1-score of 0.57 to 0.46, and `I-ORG` drops more sharply from 0.55 to 0.39. This is important because inside labels depend heavily on sequence structure. An `I-ORG` label usually only makes sense after a `B-ORG` or another `I-ORG`, but this SVM still classifies each token independently. Word embeddings can tell us something about the meaning of a word, but they do not directly tell the model whether the word is continuing a previous entity. This makes the model weaker at handling multi-word entities.

The result for `I-PER` is more complicated. Its f1-score improves from 0.48 to 0.54, but the reason is not simply that the embedding model is better overall. In part d, `I-PER` had very low precision, 0.33, but very high recall, 0.87. That means the part d classifier overpredicted `I-PER`: it found many true `I-PER` tokens but also incorrectly labeled many other tokens as `I-PER`. In part e, precision improves to 0.59, but recall drops to 0.50. This means the embedding model is less likely to overuse `I-PER`, but it also misses more true inside-person tokens. The f1-score improves because the balance between precision and recall is better.

Overall, the embedding model improves some important entity labels, especially the beginnings of person and organization names, but it does not solve the deeper problem of named entity recognition. NERC is not just about individual word meaning, it also depends on surrounding context and valid BIO label sequences. The embedding model gives the classifier better semantic information, but because the classifier is still a token-level SVM, it cannot fully model how labels depend on each other.

## [Points: 10] Exercise 2 (NERC): feature inspection using the [Annotated Corpus for Named Entity Recognition](https://www.kaggle.com/abhinavwalia95/entity-annotated-corpus)
**[6 points] a. Perform the same steps as in the previous exercise. Make sure you end up for both the training part (*df_train*) and the test part (*df_test*) with:**
* the features representation using **DictVectorizer**
* the NERC labels in a list

Please note that this is the same setup as in the previous exercise:
* load both train and test using:
    * list of dictionaries for features
    * list of NERC labels
* combine train and test features in a list and represent them using one hot encoding
* train using the training features and NERC labels

In [ ]:
import pandas

In [ ]:
##### Adapt the path to point to your local copy of NERC_datasets (see Lab4a.1-NERC-introduction.ipynb)
path = '/Users/piek/Desktop/ONDERWIJS/data/nerc_datasets/kaggle/ner_v2.csv'
kaggle_dataset = pandas.read_csv(path, on_bad_lines='skip', encoding='unicode_escape')

In [ ]:
len(kaggle_dataset)

In [ ]:
df_train = kaggle_dataset[:100000]
df_test = kaggle_dataset[100000:120000]
print(len(df_train), len(df_test))

**[4 points] b. Train and evaluate the model and provide the classification report:**
* use the SVM to predict NERC labels on the test data
* evaluate the performance of the SVM on the test data

Analyze the performance per NERC label.

## End of this notebook